# Suitability Model for Predicting Where to Plant Arabica

**Project:** LaoAgri-Suitability (LAOS)  
**Study area:** Lao PDR  
**Input:** `LaoAgri_Arabica_Environmental_Features_V2.csv`  
**Target:** `target_y` — provisional Arabica suitability index, 0–100 points  
**Scoring version:** V3 — revised score discrimination + 40% Thermal–Altitude group  
**Workflow:** FAO-informed component scoring → weighted geometric aggregation → spatially separated ML regression → prediction map

---

## ຈຸດປະສົງ

Notebook ນີ້ນຳ Dataset V2 ທີ່ສ້າງສຳເລັດແລ້ວມາສ້າງ `target_y` ລະດັບ 0–100, ຈາກນັ້ນຝຶກ Regression Models ເພື່ອຮຽນຮູ້ຈຳລອງດັດຊະນີນັ້ນ.

**ສຳຄັນ:** `target_y` ແມ່ນ *rule-based pseudo-target / screening index* ທີ່ເຮົາສ້າງຈາກ Environmental Features ຂອງ Dataset. ML Metrics ຈຶ່ງວັດວ່າ Model ຈຳລອງສູດນີ້ໄດ້ດີປານໃດ, ບໍ່ແມ່ນຫຼັກຖານວ່າດິນຈະໃຫ້ຜົນຜະລິດ Arabica ສູງຈິງ.

## ການໃຊ້ FAO ໃຫ້ຖືກບົດບາດ

- **FAO GAUL 2024** ໃຊ້ເປັນ Administrative Boundary ຂອງ Lao PDR ໃນ Dataset V2. GAUL 2024 ບໍ່ໄດ້ໃຫ້ Coffee Suitability Weights.
- **FAO Arabica Coffee Manual for Lao PDR (2005)** ເປັນແຫຼ່ງ Agronomic ຫຼັກສຳລັບ elevation, temperature, rainfall, soil pH, slope ແລະ aspect.
- **FAO EcoCrop – Coffea arabica** ໃຊ້ປະກອບຂອບເຂດ absolute ບາງຕົວ, ບໍ່ແມ່ນສູດ Weighting.
- ນ້ຳໜັກ, ບາງ Curve Breakpoints ແລະການປະສົມຄະແນນໃນ Notebook ນີ້ແມ່ນ **researcher-assigned assumptions** ທີ່ຕ້ອງລາຍງານ ແລະກວດດ້ວຍ Independent Validation.

**Dataset provenance:** FAO GAUL 2024 · Copernicus DEM GLO-30 `2024_1` · WorldClim 2.1 (1970–2000) · SoilGrids 2.0. BIO1 ໃນ Dataset V2 ຢູ່ໃນ °C ແລ້ວ; ຫ້າມຄູນ 0.1 ຊ້ຳອີກ.

## 1. Revised scoring methodology — V3

### 1.1 ນ້ຳໜັກລະດັບ Group

ເຮົາກຳນົດໃຫ້ **Elevation + Temperature = 40%** ຕາມການຕັດສິນຂອງໂຄງການ. ເພາະສອງຕົວແປມີ Correlation ສູງ, ຈຶ່ງຈັດເປັນກຸ່ມດຽວ ແລະບໍ່ໃຫ້ຄະແນນລວມຂອງກຸ່ມເກີນ 40%.

| Group | Factor | Weight | Rationale |
|---|---|---:|---|
| Thermal–Altitude (40%) | Temperature | **28%** | Direct climatic indicator; 70% ຂອງ Group |
| Thermal–Altitude (40%) | Elevation | **12%** | Highland context; 30% ຂອງ Group |
| Water (25%) | Annual rainfall | **20%** | ປະລິມານນ້ຳຝົນລວມ |
| Water (25%) | BIO14 | **5%** | Driest-month proxy ສຳລັບ dry-stress; ນ້ຳໜັກຕ່ຳເພາະວັດໄດ້ພຽງເດືອນດຽວ |
| Soil (20%) | Soil pH | **15%** | Soil reaction; ອ້າງອີງຊ່ວງ pH ທີ່ກາເຟມັກ |
| Soil (20%) | SOC | **5%** | Relative soil-carbon indicator ເທົ່ານັ້ນ |
| Terrain (15%) | Slope | **12%** | Erosion/management risk |
| Terrain (15%) | Aspect | **3%** | East/South orientation; supporting factor |
| Diagnostic only | BIO15 | **0%** | ບໍ່ມີທິດທາງ optimum ທີ່ຊັດເຈນຈາກຫຼັກຖານທີ່ເຮົາມີ; ເກັບເປັນ diagnostic ແລະ ML predictor ແຕ່ບໍ່ໃຫ້ຄະແນນ Target |
| **Total active weights** | | **100%** | BIO15 ບໍ່ຖືກນັບໃນຜົນລວມ |

### 1.2 ແກ້ Score Saturation ແນວໃດ?

ບໍ່ຄວນບັງຄັບປັບຄະແນນໃຫ້ກະຈາຍສວຍງາມ ຖ້າຂໍ້ມູນຈິງຢູ່ໃນຊ່ວງທີ່ເໝາະສົມ. V3 ຈຶ່ງແກ້ແບບນີ້:

1. Soil pH: ໃຊ້ fuzzy peak ຮອບ pH 5.5–6.0 ແທນການໃຫ້ 100 ແບບກວ້າງທັງ 5–6. ຄ່າໃກ້ optimum ຍັງໄດ້ຄະແນນສູງ, ແຕ່ບໍ່ໃຫ້ທຸກ cell ເທົ່າກັນ.
2. SOC: ບໍ່ໃຊ້ threshold `>12 g/kg = 100` ອີກ. FAO Lao manual ລາຍງານ Soil Organic Matter (OM), ແຕ່ SOC 0–30 cm ໃນ SoilGrids ບໍ່ແມ່ນການວັດດ້ວຍ depth/method ດຽວກັນ. ຈຶ່ງໃຊ້ Percentile Rank ພາຍໃນ Laos ເປັນ relative score (ບໍ່ແມ່ນ absolute agronomic suitability threshold).
3. BIO15: ບໍ່ໃຫ້ຄະແນນທິດທາງສູງ/ຕ່ຳໂດຍບໍ່ມີຫຼັກຖານ; ໃຊ້ເປັນ diagnostic ເທົ່ານັ້ນ.
4. Aggregation: ໃຊ້ **Weighted Geometric Mean** ເພື່ອຫຼຸດການຊົດເຊີຍກັນແບບເຕັມທີ່. ປັດໄຈທີ່ອ່ອນຈະກົດຄະແນນລວມໄດ້ຫຼາຍກວ່າ WLC.
5. ຄະແນນ component 0 ຖືກຍົກພື້ນເປັນ 1 ກ່ອນ log aggregation. `SCORE_FLOOR=1` ແມ່ນ numerical/modeling assumption ເພື່ອບໍ່ໃຫ້ proxy ຕົວດຽວກຳນົດ Target ເປັນ 0 ທັນທີ; ມີ Floor Sensitivity Test ໃຫ້ກວດ.

### 1.3 Suitability score bands ທີ່ຜູ້ໃຊ້ເລືອກ

| Band | Score range |
|---|---:|
| Low | **0–49** |
| Fair | **50–69** |
| Good | **70–84** |
| Very Good | **85–100** |

ໃນຕົວເລກທົດສະນິຍົມ ໃຊ້ `[0,50)`, `[50,70)`, `[70,85)`, `[85,100]`. ນີ້ແມ່ນ Project-defined index bands, ບໍ່ແມ່ນ FAO S1/S2/S3/N ຫຼືສັດສ່ວນຂອງພື້ນທີ່ທີ່ຕ້ອງບັງຄັບໃຫ້ເທົ່າກັນ.

In [ ]:
# 2. ຕິດຕັ້ງ Package ແລະ Import Libraries
# ໃນ Google Colab ໃຫ້ Run Cell ນີ້ກ່ອນ Cell ອື່ນ.

%pip -q install --upgrade xgboost joblib

import os, json, math, warnings, zipfile, joblib, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)
print("Python:", platform.python_version())
print("pandas:", pd.__version__)

In [ ]:
# 3. ເຊື່ອມ Google Drive ແລະກຳນົດ Path
# Path ນີ້ອ້າງອີງຕາມທີ່ລະບຸໃນ LaoAgri ML Handoff.

from google.colab import drive
from google.colab import files

drive.mount('/content/drive')

DATA_PATH = Path('/content/drive/MyDrive/LaoAgri_V2/outputs/LaoAgri_Arabica_Environmental_Features_V2.csv')
OUTPUT_DIR = Path('/content/drive/MyDrive/LaoAgri_Suitability_Model_Outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Dataset exists:', DATA_PATH.exists())
print('Output folder:', OUTPUT_DIR)

# ຖ້າ Path ໃນ Drive ບໍ່ກົງກັນ ສາມາດແກ້ DATA_PATH ດ້ານເທິງ
# ຫຼື Upload CSV ດ້ວຍ Cell ຖັດໄປ.

In [ ]:
# 4. Load Dataset V2
# ບໍ່ມີການສ້າງ AOI ຫຼື Raster ໃໝ່ໃນ Notebook ນີ້.
# ພວກເຮົາໃຊ້ CSV ທີ່ສ້າງຈາກ FAO GAUL 2024 + Dataset V2 ແລ້ວ.

if not DATA_PATH.exists():
    print('ບໍ່ພົບ CSV ຕາມ Drive Path. ກະລຸນາ Upload LaoAgri_Arabica_Environmental_Features_V2.csv')
    uploaded = files.upload()
    csv_candidates = [name for name in uploaded.keys() if name.lower().endswith('.csv')]
    if not csv_candidates:
        raise FileNotFoundError('ບໍ່ພົບໄຟລ໌ CSV ທີ່ Upload.')
    DATA_PATH = Path(csv_candidates[0])

df_raw = pd.read_csv(DATA_PATH)
print('Dataset shape:', df_raw.shape)
display(df_raw.head())

In [ ]:
# 5. ตรวจสอบ Schema ແລະ Data Quality ຂັ້ນຕົ້ນ
# ກວດວ່າ Dataset V2 ມີ 9 Environmental Features ຕາມ Handoff.

REQUIRED_COLUMNS = [
    'grid_id', 'lon', 'lat',
    'elevation_m', 'temperature_c', 'rainfall_mm_year', 'soil_ph',
    'slope_pct', 'bio14_driest_month_mm', 'bio15_seasonality_cv',
    'soc_0_30cm_g_kg', 'aspect_deg', 'aspect_sin', 'aspect_cos', 'aspect_is_defined'
]
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]
if missing_cols:
    raise ValueError(f'Dataset ຂາດ Columns ທີ່ຈຳເປັນ: {missing_cols}')

print('Missing cells by column:')
display(df_raw[REQUIRED_COLUMNS].isna().sum().rename('missing_count').to_frame())
print('Duplicate grid_id:', int(df_raw['grid_id'].duplicated().sum()))
print('Longitude bounds:', df_raw['lon'].min(), 'to', df_raw['lon'].max())
print('Latitude bounds:', df_raw['lat'].min(), 'to', df_raw['lat'].max())

# ກວດ units/value ranges ເພື່ອຈັບຄວາມຜິດພາດໃຫຍ່ໆ.
range_checks = {
    'temperature_c': (-5, 40),
    'rainfall_mm_year': (0, 10000),
    'soil_ph': (0, 14),
    'slope_pct': (0, 300),
    'bio14_driest_month_mm': (0, 1000),
    'bio15_seasonality_cv': (0, 500),
    'soc_0_30cm_g_kg': (0, 200),
}
for col, (lo, hi) in range_checks.items():
    values = pd.to_numeric(df_raw[col], errors='coerce')
    if values.notna().any():
        print(f'{col:28s} min={values.min():.3f} | max={values.max():.3f} | expected broad range=({lo},{hi})')
        if (values.dropna() < lo).any() or (values.dropna() > hi).any():
            print('  WARNING: ພົບຄ່ານອກ broad range — ກະລຸນາກວດ Dataset ກ່ອນຕີຄວາມຜົນ.')

## 2. ການຈັດການ Missing Values

ໃນ Dataset V2 ອາດມີ `soil_ph` ຫຼື `soc_0_30cm_g_kg` ບາງ cell ບໍ່ມີຄ່າ. ເພື່ອບໍ່ໃຫ້ຫາຍ grid cells ຈາກຜົນສະແດງລະດັບປະເທດ, ຂັ້ນຕອນນີ້ຈະ:

- ເກັບຈຳນວນ Missing ໄວ້ເປັນ Audit.
- ແທນ Missing pH/SOC ດ້ວຍ Median ຂອງຄ່າທີ່ມີໃນ Dataset ສຳລັບຄຳນວນ Screening Score.
- ເພີ່ມ Flag `soil_ph_was_missing` ແລະ `soc_was_missing` ໃນ Output ເພື່ອບອກບ່ອນທີ່ມີການ Impute.
- Aspect ທີ່ Undefined ບໍ່ຖືກຕັດອອກ; ໃຫ້ Neutral Aspect Score ແທນ.

ການ Impute ນີ້ເປັນການຈັດການທາງຄຳນວນ, ບໍ່ແມ່ນການຢືນຢັນຄ່າດິນຈິງໃນ cell ນັ້ນ.

In [ ]:
# 6. ສ້າງ Work Dataset ແລະຈັດການ Missing Values

work_df = df_raw.copy()
work_df['soil_ph_was_missing'] = work_df['soil_ph'].isna().astype('int8')
work_df['soc_was_missing'] = work_df['soc_0_30cm_g_kg'].isna().astype('int8')

PH_MEDIAN = float(work_df['soil_ph'].median())
SOC_MEDIAN = float(work_df['soc_0_30cm_g_kg'].median())
work_df['soil_ph'] = work_df['soil_ph'].fillna(PH_MEDIAN)
work_df['soc_0_30cm_g_kg'] = work_df['soc_0_30cm_g_kg'].fillna(SOC_MEDIAN)

# ຄ່າ aspect_deg ສາມາດສ້າງຄືນຈາກ sin/cos ໄດ້ ສຳລັບ cell ທີ່ aspect_is_defined=1.
missing_angle = work_df['aspect_deg'].isna() & (work_df['aspect_is_defined'].astype(int) == 1)
if missing_angle.any():
    reconstructed = (np.degrees(np.arctan2(work_df.loc[missing_angle, 'aspect_sin'],
                                           work_df.loc[missing_angle, 'aspect_cos'])) + 360) % 360
    work_df.loc[missing_angle, 'aspect_deg'] = reconstructed

# ກວດ Missing ໃນຄ່າຈຳເປັນອື່ນໆ; ຖ້າມີໃຫ້ຢຸດເພື່ອບໍ່ໃຫ້ target ຄຳນວນດ້ວຍຄ່າຜິດ.
core_numeric = [
    'lon', 'lat', 'elevation_m', 'temperature_c', 'rainfall_mm_year',
    'soil_ph', 'slope_pct', 'bio14_driest_month_mm', 'bio15_seasonality_cv',
    'soc_0_30cm_g_kg', 'aspect_sin', 'aspect_cos', 'aspect_is_defined'
]
if work_df[core_numeric].isna().any().any():
    bad = work_df[core_numeric].isna().sum()
    raise ValueError(f'ຍັງມີ Missing values ໃນ columns ທີ່ຈຳເປັນ:\n{bad[bad > 0]}')

print('pH median used for imputation:', PH_MEDIAN)
print('SOC median used for imputation:', SOC_MEDIAN)
print('pH imputed rows:', int(work_df['soil_ph_was_missing'].sum()))
print('SOC imputed rows:', int(work_df['soc_was_missing'].sum()))
print('Rows retained:', len(work_df))

## 3. Component scoring curves (0–100)

ແຕ່ລະ Feature ຈະຖືກແປງເປັນ Component Score 0–100 ດ້ວຍ Piecewise-linear fuzzy membership function. ຈຸດທີ່ FAO ລະບຸໂດຍກົງ ແລະຈຸດທີ່ເຮົາສົມມຸດເພື່ອສ້າງ curve ຈະຖືກແຍກໃຫ້ຊັດ.

- **Elevation:** FAO Lao manual ລະບຸວ່າ >1,000 m ເໝາະກັບ Arabica ຄຸນນະພາບສູງ. Curve ໃຫ້ຄະແນນຄ່ອຍໆສູງຈາກ 400–1,000 m ແລະຮັກສາຄະແນນສູງເຖິງ 2,800 m (FAO EcoCrop absolute altitude ceiling). 400/800 m ແມ່ນ interpolation anchors ຂອງ project, ບໍ່ແມ່ນ FAO hard limits.
- **Temperature:** FAO Lao manual ລາຍງານ mean temperature ຂອງ Bolaven sites 19.0–22.2°C, ລະບຸວ່າ <15°C ຈຳກັດການເຕີບໂຕ ແລະ >30°C ເກີດ stress. ຂໍ້ມູນ V2 ແມ່ນ WorldClim BIO1 (annual mean), ບໍ່ແມ່ນ daily temperature; ການປຽບທຽບນີ້ເປັນ proxy.
- **Annual rainfall:** FAO Lao manual ກ່າວວ່າ ideal rainfall >1,200–1,500 mm/year ແລະລາຍງານ Bolaven sites 2,500–3,534 mm/year. Curve ໃຫ້ຄະແນນສູງແກ່ local high-rainfall range; 1,200/1,500/2,500/3,534 ເປັນ anchors ທີ່ອ້າງອີງຈາກຄູ່ມື, ຈຸດລະຫວ່າງກາງເປັນ interpolation assumption.
- **Soil pH:** FAO Lao manual ລະບຸວ່າ coffee prefers pH 5–6 ແລະຕາຕະລາງ optimum soil pH 5.5–6.0 (ວິທີ pH 1:5). Curve ສ້າງ peak ຮອບ 5.75 ແລະຫຼຸດລົງຢ່າງຄ່ອຍໆ; SoilGrids pH H2O ແລະ depth 0–30 cm ບໍ່ໄດ້ກົງກັບຕົວຢ່າງດິນຂອງຄູ່ມືຢ່າງສົມບູນ.
- **Slope:** FAO Lao manual ລະບຸວ່າ slope <15% ມັກດີກວ່າ ແລະ slope ຊັນເພີ່ມ erosion risk. ຈຸດຫຼຸດຄະແນນຫຼັງ 15% ຕາມ curve ແມ່ນ modeling assumptions; slope V2 ແມ່ນ mean ໃນ grid cell ~928 m, ບໍ່ແມ່ນ plot slope.
- **BIO14:** ໃຊ້ precipitation ຂອງເດືອນແຫ້ງທີ່ສຸດ. FAO manual ອະທິບາຍວ່າ coffee ຕ້ອງການ dry stress period ເພື່ອຊ່ວຍ flowering, ແຕ່ BIO14 ບອກພຽງ 1 ເດືອນ ບໍ່ໄດ້ບອກຈຳນວນເດືອນແຫ້ງ. ຈຶ່ງໃຫ້ Weight ຕ່ຳ 5% ແລະຖື curve ເປັນ proxy assumption.
- **SOC:** ໃຊ້ percentile rank ພາຍໃນ Laos ເພາະບໍ່ມີ validated Arabica SOC threshold ທີ່ກົງກັບ SoilGrids SOC 0–30 cm. ຄະແນນນີ້ແມ່ນ relative ranking, ບໍ່ແມ່ນ absolute soil quality.
- **Aspect:** FAO Lao manual ລະບຸວ່າ east-facing ຫຼື south-facing aspect ເໝາະກວ່າ. Undefined/flat aspect ໄດ້ neutral score 75.
- **BIO15:** ເກັບເປັນ diagnostic/model predictor ເທົ່ານັ້ນ. ຄ່າ CV ສູງ/ຕ່ຳຢ່າງດຽວບໍ່ບອກວ່າ seasonality ເໝາະສົມກັບ coffee ຫຼືບໍ່.

**ຫ້າມປັບ curve ພຽງເພື່ອໃຫ້ histogram ກະຈາຍສວຍ.** ຕ້ອງປັບເພາະມີ Agronomic Rationale ຫຼືເປັນ Relative Indicator ທີ່ລະບຸຂໍ້ຈຳກັດຊັດເຈນ.

In [ ]:
# 7. ກຳນົດ Weights ແລະ Scoring Functions — V3
# Active factors ມີ 8 ຕົວ; BIO15 ເປັນ diagnostic-only (weight = 0).

THERMAL_ALTITUDE_GROUP_WEIGHT = 40.0
TEMPERATURE_SHARE_IN_GROUP = 0.70
ELEVATION_SHARE_IN_GROUP = 0.30

# ກຸ່ມອື່ນມີ Weight ລວມ 60%; ການແບ່ງແມ່ນ project assumptions.
OTHER_FACTOR_BASE_WEIGHTS = {
    'rainfall': 20.0,
    'bio14': 5.0,
    'soil_ph': 15.0,
    'soc': 5.0,
    'slope': 12.0,
    'aspect': 3.0,
}


def build_weights(thermal_group_weight=40.0, temperature_share=0.70):
    """ສ້າງ Weight ໂດຍຮັກສາຜົນລວມເປັນ 100%.

    temperature_share ແມ່ນສັດສ່ວນ Temperature ພາຍໃນ Thermal–Altitude Group.
    OTHER_FACTOR_BASE_WEIGHTS ຖືກ normalize ໃຫ້ຮັບນ້ຳໜັກທີ່ເຫຼືອ.
    """
    if not 0 < thermal_group_weight < 100:
        raise ValueError('thermal_group_weight ຕ້ອງຢູ່ລະຫວ່າງ 0 ແລະ 100.')
    if not 0 < temperature_share < 1:
        raise ValueError('temperature_share ຕ້ອງຢູ່ລະຫວ່າງ 0 ແລະ 1.')
    other_total = sum(OTHER_FACTOR_BASE_WEIGHTS.values())
    other_scale = (100.0 - thermal_group_weight) / other_total
    result = {
        'elevation': thermal_group_weight * (1.0 - temperature_share),
        'temperature': thermal_group_weight * temperature_share,
    }
    result.update({k: v * other_scale for k, v in OTHER_FACTOR_BASE_WEIGHTS.items()})
    return result


WEIGHTS = build_weights(THERMAL_ALTITUDE_GROUP_WEIGHT, TEMPERATURE_SHARE_IN_GROUP)
WEIGHTED_FACTORS = list(WEIGHTS.keys())

assert abs(sum(WEIGHTS.values()) - 100.0) < 1e-8
assert abs(WEIGHTS['elevation'] + WEIGHTS['temperature'] - 40.0) < 1e-8
assert abs(WEIGHTS['temperature'] - 28.0) < 1e-8
assert abs(WEIGHTS['elevation'] - 12.0) < 1e-8


def interp_score(values, x_points, y_points):
    """Piecewise-linear score clipped to 0–100."""
    return np.interp(np.asarray(values, dtype=float), x_points, y_points).clip(0, 100)


# Elevation (m): transition into highland suitability; >1,000 m preferred in FAO Lao manual.
def score_elevation(x):
    return interp_score(x,
        [0, 400, 800, 1000, 2800, 3000],
        [0,   0,  60,  100,  100,    0])


# WorldClim BIO1 annual mean temperature (°C): selected Bolaven means 19.0–22.2°C.
def score_temperature(x):
    return interp_score(x,
        [0, 12, 15, 19, 22.2, 25, 28, 30, 40],
        [0,  0,  0,100,  100, 75, 30,  0,  0])


# Annual rainfall (mm/year): preserve high scores for documented high-rainfall Bolaven locations.
def score_rainfall(x):
    return interp_score(x,
        [0, 750, 1200, 1500, 2500, 3534, 4200, 5000],
        [0,   0,   60,   85,  100,  100,    0,    0])


# Soil pH: fuzzy peak around 5.75; broad good range remains around pH 5–6.
def score_soil_ph(x):
    return interp_score(x,
        [0, 4.3, 4.7, 5.0, 5.5, 5.75, 6.0, 6.5, 7.0, 8.4, 14],
        [0,   0,  40,  85,  95,  100,  95,  70,  40,   0,  0])


# Slope (%): <15% preferred; the decline and zero endpoint are project assumptions.
def score_slope(x):
    return interp_score(x,
        [0, 15, 30, 45, 60, 100, 200],
        [100,100, 70, 35,  0,   0,   0])


# BIO14 (driest month rainfall, mm): low weight proxy for a seasonal dry stress period.
def score_bio14(x):
    return interp_score(x,
        [0, 5, 15, 25, 40, 60, 100],
        [100,100,80, 60, 25,  0,   0])


# SOC (g/kg): rank-based relative indicator, not an absolute agronomic threshold.
def score_soc_relative(values, missing_mask=None):
    values = pd.Series(np.asarray(values, dtype=float))
    scores = values.rank(method='average', pct=True) * 100.0
    if missing_mask is not None:
        scores.loc[np.asarray(missing_mask, dtype=bool)] = 50.0  # neutral score for imputed/missing SOC
    return scores.to_numpy(dtype=float)


# Aspect: East/South-facing directions receive higher score; undefined aspect is neutral.
def score_aspect(aspect_deg, aspect_is_defined):
    aspect = np.asarray(aspect_deg, dtype=float)
    defined = np.asarray(aspect_is_defined).astype(bool)
    d_east = np.abs((aspect - 90 + 180) % 360 - 180)
    d_south = np.abs((aspect - 180 + 180) % 360 - 180)
    best_cosine = np.maximum(np.cos(np.deg2rad(d_east)), np.cos(np.deg2rad(d_south)))
    scores = 50 + 50 * best_cosine
    scores = np.where(defined & np.isfinite(aspect), scores, 75.0)
    return np.clip(scores, 0, 100)


# Numerical floor for weighted geometric aggregation; tested later with 1/5/10-point alternatives.
SCORE_FLOOR = 1.0

# Display active weights; BIO15 is deliberately diagnostic-only.
weights_table = pd.DataFrame([
    {'factor': k, 'weight_percent': v,
     'group': ('Thermal–Altitude' if k in ['elevation','temperature'] else
               'Water' if k in ['rainfall','bio14'] else
               'Soil' if k in ['soil_ph','soc'] else 'Terrain')}
    for k, v in WEIGHTS.items()
])
display(weights_table)
print('Active weight total:', round(weights_table['weight_percent'].sum(), 6))
print('Thermal–Altitude group:', round(WEIGHTS['elevation'] + WEIGHTS['temperature'], 2), '%')
print('BIO15 weight: 0% (diagnostic-only; retained as an ML predictor).')

In [ ]:
# 8. ຄຳນວນ Component Scores ແລະ Target y 0–100 — Weighted Geometric Mean

COMPONENT_FUNCTIONS = {
    'score_elevation': ('elevation_m', score_elevation),
    'score_temperature': ('temperature_c', score_temperature),
    'score_rainfall': ('rainfall_mm_year', score_rainfall),
    'score_soil_ph': ('soil_ph', score_soil_ph),
    'score_slope': ('slope_pct', score_slope),
    'score_bio14': ('bio14_driest_month_mm', score_bio14),
}

for score_col, (source_col, func) in COMPONENT_FUNCTIONS.items():
    work_df[score_col] = func(work_df[source_col].to_numpy())

work_df['score_soc'] = score_soc_relative(
    work_df['soc_0_30cm_g_kg'].to_numpy(),
    missing_mask=work_df['soc_was_missing'].to_numpy()
)
work_df['score_aspect'] = score_aspect(
    work_df['aspect_deg'].to_numpy(),
    work_df['aspect_is_defined'].to_numpy()
)

# ກວດໃຫ້ແນ່ໃຈວ່າ Score Matrix ແລະ Weight ມີລຳດັບດຽວກັນ.
SCORE_COLUMN_BY_FACTOR = {
    'elevation': 'score_elevation',
    'temperature': 'score_temperature',
    'rainfall': 'score_rainfall',
    'soil_ph': 'score_soil_ph',
    'slope': 'score_slope',
    'bio14': 'score_bio14',
    'soc': 'score_soc',
    'aspect': 'score_aspect',
}
component_cols = list(SCORE_COLUMN_BY_FACTOR.values())
score_matrix = work_df[component_cols].to_numpy(dtype='float64')
weight_vector = np.array([WEIGHTS[k] for k in SCORE_COLUMN_BY_FACTOR], dtype='float64') / 100.0
assert np.isclose(weight_vector.sum(), 1.0)

# WLC ເກັບໄວ້ເປັນ comparison/diagnostic; ບໍ່ແມ່ນ Target ຫຼັກ.
work_df['target_y_wlc'] = np.sum(score_matrix * weight_vector, axis=1).clip(0, 100)

# Weighted Geometric Mean: less compensatory than arithmetic WLC.
# Floor=1 prevents one zero-valued coarse-scale proxy from forcing the entire index to zero.
score_matrix_for_log = np.maximum(score_matrix, SCORE_FLOOR) / 100.0
work_df['target_y'] = 100.0 * np.exp(np.sum(np.log(score_matrix_for_log) * weight_vector, axis=1))
work_df['target_y'] = work_df['target_y'].clip(0, 100)

# User-selected project score bands: maximum integer points 49 / 69 / 84 / 100.
BAND_LABELS = ['Low', 'Fair', 'Good', 'Very Good']
BAND_BINS = [0.0, 50.0, 70.0, 85.0, 100.000001]

def classify_suitability(values):
    return pd.cut(
        np.asarray(values, dtype=float),
        bins=BAND_BINS,
        labels=BAND_LABELS,
        right=False,
        include_lowest=True
    ).astype(str)

work_df['suitability_band'] = classify_suitability(work_df['target_y'])
work_df['limiting_feature'] = (
    work_df[component_cols].idxmin(axis=1).str.replace('score_', '', regex=False)
)

assert abs(sum(WEIGHTS.values()) - 100.0) < 1e-8
assert work_df['target_y'].between(0, 100).all()
assert work_df['suitability_band'].notna().all()

print('Target y (Weighted Geometric Mean) summary:')
display(work_df['target_y'].describe(percentiles=[.01,.05,.10,.25,.50,.75,.90,.95,.99]).to_frame())
print('WLC comparison summary:')
display(work_df['target_y_wlc'].describe().to_frame())
print('Suitability band distribution:')
band_distribution = (
    work_df['suitability_band'].value_counts()
    .reindex(BAND_LABELS, fill_value=0).rename('cells').to_frame()
)
band_distribution['percent'] = band_distribution['cells'] / len(work_df) * 100

display(band_distribution)
band_distribution.to_csv(OUTPUT_DIR / 'suitability_class_distribution.csv')

## 3.1 ປຽບທຽບ Thermal–Altitude Group Weight

ຕາມການຕັດສິນຂອງໂຄງການ, Baseline ໃຫ້ Elevation + Temperature ລວມກັນ 40%. ເພື່ອບໍ່ລັອກຄ່າ 40% ໂດຍບໍ່ມີ sensitivity check, ຈະປຽບທຽບ 35%, 40% ແລະ 45%. ພາຍໃນກຸ່ມແບ່ງ Temperature 70% ແລະ Elevation 30% ຄົງທີ່. ນ້ຳໜັກຂອງ factors ອື່ນຖືກ scale ໃຫ້ລວມເປັນ 100% ໃນແຕ່ລະ scenario.

ການປຽບທຽບນີ້ກວດວ່າການເພີ່ມ/ຫຼຸດ group weight ປ່ຽນ ranking ແລະ class distribution ແນວໃດ. ມັນບໍ່ແມ່ນ validation ກັບການປູກຈິງ.

In [ ]:
# 8B. Thermal–Altitude group weight sensitivity: 35%, 40%, 45%
from scipy.stats import spearmanr

SCORE_FLOOR_FOR_SCENARIOS = SCORE_FLOOR
BASELINE_SCORE = work_df['target_y'].to_numpy(dtype='float64')
BASELINE_BAND = classify_suitability(BASELINE_SCORE)


def calculate_geometric_scenario(weights, floor=SCORE_FLOOR_FOR_SCENARIOS):
    matrix = np.column_stack([
        work_df[SCORE_COLUMN_BY_FACTOR[f]].to_numpy(dtype='float64')
        for f in WEIGHTED_FACTORS
    ])
    w = np.array([weights[f] for f in WEIGHTED_FACTORS], dtype='float64') / 100.0
    if not np.isclose(w.sum(), 1.0):
        raise ValueError('Scenario weights must sum to 100%.')
    return 100.0 * np.exp(np.sum(np.log(np.maximum(matrix, floor) / 100.0) * w, axis=1))


THERMAL_GROUP_SCENARIOS = [35.0, 40.0, 45.0]
scenario_rows = []
for group_weight in THERMAL_GROUP_SCENARIOS:
    scenario_weights = build_weights(group_weight, TEMPERATURE_SHARE_IN_GROUP)
    scenario_score = calculate_geometric_scenario(scenario_weights)
    scenario_band = classify_suitability(scenario_score)
    scenario_rows.append({
        'thermal_altitude_group_weight_pct': group_weight,
        'elevation_weight_pct': scenario_weights['elevation'],
        'temperature_weight_pct': scenario_weights['temperature'],
        'mean_score': float(np.mean(scenario_score)),
        'median_score': float(np.median(scenario_score)),
        'std_score': float(np.std(scenario_score)),
        'low_band_pct': float(np.mean(scenario_band == 'Low') * 100),
        'fair_band_pct': float(np.mean(scenario_band == 'Fair') * 100),
        'good_band_pct': float(np.mean(scenario_band == 'Good') * 100),
        'very_good_band_pct': float(np.mean(scenario_band == 'Very Good') * 100),
        'spearman_rank_vs_baseline': float(spearmanr(BASELINE_SCORE, scenario_score).statistic),
        'mean_abs_score_change': float(np.mean(np.abs(scenario_score - BASELINE_SCORE))),
        'cells_changed_band_pct': float(np.mean(scenario_band != BASELINE_BAND) * 100),
    })

thermal_weight_sensitivity_df = pd.DataFrame(scenario_rows)
display(thermal_weight_sensitivity_df.round(3))
thermal_weight_sensitivity_df.to_csv(OUTPUT_DIR / 'thermal_altitude_weight_sensitivity.csv', index=False)

# Floor sensitivity: check the influence of the numerical lower floor in geometric aggregation.
floor_rows = []
for floor_value in [1.0, 5.0, 10.0]:
    floor_score = calculate_geometric_scenario(WEIGHTS, floor=floor_value)
    floor_band = classify_suitability(floor_score)
    floor_rows.append({
        'score_floor': floor_value,
        'mean_score': float(np.mean(floor_score)),
        'median_score': float(np.median(floor_score)),
        'min_score': float(np.min(floor_score)),
        'max_score': float(np.max(floor_score)),
        'low_band_pct': float(np.mean(floor_band == 'Low') * 100),
        'fair_band_pct': float(np.mean(floor_band == 'Fair') * 100),
        'good_band_pct': float(np.mean(floor_band == 'Good') * 100),
        'very_good_band_pct': float(np.mean(floor_band == 'Very Good') * 100),
        'spearman_rank_vs_floor1': float(spearmanr(BASELINE_SCORE, floor_score).statistic),
        'mean_abs_change_vs_floor1': float(np.mean(np.abs(floor_score - BASELINE_SCORE))),
    })
floor_sensitivity_df = pd.DataFrame(floor_rows)
display(floor_sensitivity_df.round(3))
floor_sensitivity_df.to_csv(OUTPUT_DIR / 'geometric_score_floor_sensitivity.csv', index=False)
print('Baseline target_y uses SCORE_FLOOR =', SCORE_FLOOR)

## 3.2 Weight Sensitivity ສຳລັບ Factors ນອກ Thermal–Altitude Group

Thermal–Altitude Group ຖືກລັອກໄວ້ 40% (Elevation 12%, Temperature 28%). ໃນການທົດສອບນີ້ຈະປ່ຽນ Weight ຂອງ factor ອື່ນໆທີລະ 1 ຕົວ ±20%, ແລ້ວ normalize ກຸ່ມອື່ນໃຫ້ຍັງລວມເປັນ 60%. ດັ່ງນັ້ນ Thermal–Altitude Group ຍັງຄົງ 40% ທຸກ scenario.

ລາຍງານ Mean Absolute Score Change, Spearman Rank Correlation ແລະສັດສ່ວນຂອງ cells ທີ່ປ່ຽນ Band. ຜົນນີ້ເປັນ sensitivity analysis, ບໍ່ແມ່ນ validation.

In [ ]:
# 8C. One-at-a-time +/-20% sensitivity for non-thermal factors
from scipy.stats import spearmanr

BASELINE_SCORE = work_df['target_y'].to_numpy(dtype='float64')
BASELINE_BAND = classify_suitability(BASELINE_SCORE)
NON_THERMAL_FACTORS = list(OTHER_FACTOR_BASE_WEIGHTS.keys())
SENSITIVITY_ROWS = []

for factor in NON_THERMAL_FACTORS:
    for multiplier, scenario_name in [(0.8, 'minus_20pct'), (1.2, 'plus_20pct')]:
        other_weights = OTHER_FACTOR_BASE_WEIGHTS.copy()
        other_weights[factor] *= multiplier
        scale = 60.0 / sum(other_weights.values())
        scenario_weights = {
            'elevation': 12.0,
            'temperature': 28.0,
            **{k: v * scale for k, v in other_weights.items()}
        }
        scenario_score = calculate_geometric_scenario(scenario_weights)
        scenario_band = classify_suitability(scenario_score)
        SENSITIVITY_ROWS.append({
            'changed_factor': factor,
            'scenario': scenario_name,
            'original_weight_pct': WEIGHTS[factor],
            'new_normalized_weight_pct': scenario_weights[factor],
            'thermal_altitude_group_weight_pct': 40.0,
            'mean_absolute_score_change_points': float(np.mean(np.abs(scenario_score - BASELINE_SCORE))),
            'spearman_rank_correlation_vs_baseline': float(spearmanr(BASELINE_SCORE, scenario_score).statistic),
            'cells_changed_band_pct': float(np.mean(scenario_band != BASELINE_BAND) * 100.0),
        })

sensitivity_df = pd.DataFrame(SENSITIVITY_ROWS).sort_values(
    ['changed_factor', 'scenario']
).reset_index(drop=True)
display(sensitivity_df.round(3))
sensitivity_df.to_csv(OUTPUT_DIR / 'weight_sensitivity_summary.csv', index=False)
print('Saved: weight_sensitivity_summary.csv')

In [ ]:
# 9. ກວດ Component Score Saturation ແລະ Target Distribution

component_summary = work_df[component_cols].describe().T[['mean','std','min','25%','50%','75%','max']]
display(component_summary)

saturation_rows = []
for score_col in component_cols:
    values = work_df[score_col].to_numpy(dtype=float)
    saturation_rows.append({
        'component': score_col.replace('score_', ''),
        'mean': float(np.mean(values)),
        'median': float(np.median(values)),
        'pct_exactly_100': float(np.mean(values >= 99.999) * 100),
        'pct_ge_95': float(np.mean(values >= 95) * 100),
        'pct_le_5': float(np.mean(values <= 5) * 100),
        'saturation_flag_ge90pct_at_100': bool(np.mean(values >= 99.999) >= 0.90),
    })
saturation_df = pd.DataFrame(saturation_rows)
display(saturation_df.round(3))
saturation_df.to_csv(OUTPUT_DIR / 'component_score_saturation_diagnostic.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(work_df['target_y'], bins=np.arange(0, 102, 2), edgecolor='white')
for cut in [50, 70, 85]:
    axes[0].axvline(cut, linestyle='--', linewidth=1)
axes[0].set_title('V3 weighted geometric target_y distribution')
axes[0].set_xlabel('Suitability index (0–100)')
axes[0].set_ylabel('Number of grid cells')

means = work_df[component_cols].mean().sort_values()
axes[1].barh(means.index.str.replace('score_', '', regex=False), means.values)
axes[1].set_title('Mean component scores (diagnostic)')
axes[1].set_xlabel('Mean component score (0–100)')
plt.tight_layout()
plt.show()

print('Target class distribution:')
display(work_df['suitability_band'].value_counts().reindex(BAND_LABELS, fill_value=0).to_frame('cells'))

## 4. ກຳນົດ X ແລະ y

- **y:** `target_y` ຄະແນນ 0–100 ທີ່ຄຳນວນຈາກ Weighted Geometric Rule (V3).
- **X:** Environmental Predictors ທັງ 9 conceptual variables; BIO15 ເປັນ diagnostic-only ໃນ Target ແຕ່ຍັງຢູ່ໃນ X. Aspect ໃຊ້ `aspect_sin`, `aspect_cos`, `aspect_is_defined` ແທນ raw degrees ເພື່ອຮັກສາ Circular Nature ຂອງມັນ.
- `grid_id`, `lon`, `lat` ແມ່ນ identifiers/spatial metadata ບໍ່ໃຫ້ Model ໃຊ້ເປັນ Features ໃນຮຸ່ນພື້ນຖານ.

**ໝາຍເຫດ:** Aspect ໜຶ່ງ conceptual feature ຈະກາຍເປັນ 3 model columns (sin, cos, defined indicator). ດັ່ງນັ້ນ X ມີ 11 numerical columns ຈາກ 9 conceptual features.

In [ ]:
# 10. ກຳນົດ Model Features, Correlation ແລະ VIF Diagnostic

MODEL_FEATURES = [
    'elevation_m', 'temperature_c', 'rainfall_mm_year', 'soil_ph', 'slope_pct',
    'bio14_driest_month_mm', 'bio15_seasonality_cv', 'soc_0_30cm_g_kg',
    'aspect_sin', 'aspect_cos', 'aspect_is_defined'
]
TARGET_COLUMN = 'target_y'

X = work_df[MODEL_FEATURES].astype('float32')
y = work_df[TARGET_COLUMN].astype('float32')

corr_cols = [
    'elevation_m', 'temperature_c', 'rainfall_mm_year', 'soil_ph', 'slope_pct',
    'bio14_driest_month_mm', 'bio15_seasonality_cv', 'soc_0_30cm_g_kg'
]
pearson_corr = work_df[corr_cols].corr(method='pearson')
spearman_corr = work_df[corr_cols].corr(method='spearman')
print('Elevation–Temperature Pearson r:', round(float(pearson_corr.loc['elevation_m', 'temperature_c']), 4))
print('Elevation–Temperature Spearman rho:', round(float(spearman_corr.loc['elevation_m', 'temperature_c']), 4))

plt.figure(figsize=(10, 7))
plt.imshow(spearman_corr, vmin=-1, vmax=1, cmap='coolwarm')
plt.colorbar(label='Spearman correlation')
plt.xticks(range(len(corr_cols)), corr_cols, rotation=75, ha='right')
plt.yticks(range(len(corr_cols)), corr_cols)
plt.title('Environmental feature correlation (Spearman) — Dataset V2')
plt.tight_layout()
plt.show()

# VIF: regress each predictor on all other predictors. Use a reproducible sample to reduce compute.
VIF_MAX_ROWS = 50000
vif_frame = work_df[MODEL_FEATURES].sample(
    n=min(VIF_MAX_ROWS, len(work_df)), random_state=SEED
).dropna().reset_index(drop=True)
vif_rows = []
for feature in MODEL_FEATURES:
    other_features = [c for c in MODEL_FEATURES if c != feature]
    r2_aux = LinearRegression().fit(vif_frame[other_features], vif_frame[feature]).score(
        vif_frame[other_features], vif_frame[feature]
    )
    vif = np.inf if r2_aux >= 0.999999 else 1.0 / (1.0 - r2_aux)
    vif_rows.append({'feature': feature, 'auxiliary_R2': float(r2_aux), 'VIF': float(vif)})

vif_df = pd.DataFrame(vif_rows).sort_values('VIF', ascending=False).reset_index(drop=True)
vif_df['diagnostic'] = np.select(
    [vif_df['VIF'] >= 10, vif_df['VIF'] >= 5],
    ['High (>=10)', 'Review (5–10)'], default='Lower (<5)'
)
display(vif_df.round(3))
vif_df.to_csv(OUTPUT_DIR / 'vif_diagnostic.csv', index=False)

print('VIF sample rows:', len(vif_frame))
print('Interpretation: VIF diagnoses predictor redundancy; it does NOT calculate agronomic weights.')
print('X shape:', X.shape, '| y shape:', y.shape)

## 5. Spatial Train/Test Split

ຫ້າມໃຊ້ Random Row Split ແບບທົ່ວໄປເປັນການປະເມີນຫຼັກ, ເພາະ Grid Cells ທີ່ຢູ່ໃກ້ກັນອາດມີສະພາບພູມອາກາດ ແລະພູມສັນຖານຄ້າຍກັນ.

Notebook ນີ້ໃຊ້ **0.5° × 0.5° Spatial Blocks** ຈາກ Longitude/Latitude ແລ້ວແຍກ 80% ຂອງ Blocks ໄວ້ Train ແລະ 20% ເປັນ Held-out Test. ຂະໜາດ 0.5° ແມ່ນຄ່າເລີ່ມຕົ້ນທີ່ເລືອກເພື່ອຈັດການ Spatial Dependence, ບໍ່ແມ່ນຂະໜາດທີ່ພິສູດວ່າເໝາະທີ່ສຸດ. ຄວນທົດສອບ block size ອື່ນໃນວຽກຕໍ່ໄປ.

In [ ]:
# 11. ສ້າງ Spatial Block IDs ແລະແບ່ງ Train/Test ໂດຍບໍ່ໃຫ້ Block ຊ້ຳກັນ

BLOCK_SIZE_DEG = 0.5
work_df['block_lon'] = np.floor(work_df['lon'] / BLOCK_SIZE_DEG).astype(int)
work_df['block_lat'] = np.floor(work_df['lat'] / BLOCK_SIZE_DEG).astype(int)
work_df['spatial_block_id'] = (
    work_df['block_lon'].astype(str) + '_' + work_df['block_lat'].astype(str)
)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_pos, test_pos = next(splitter.split(work_df, y, groups=work_df['spatial_block_id']))

train_df = work_df.iloc[train_pos].copy().reset_index(drop=True)
test_df = work_df.iloc[test_pos].copy().reset_index(drop=True)

train_blocks = set(train_df['spatial_block_id'].unique())
test_blocks = set(test_df['spatial_block_id'].unique())
assert train_blocks.isdisjoint(test_blocks), 'Spatial leakage: blocks overlap between train and test.'

print('Train rows:', len(train_df), '| Train blocks:', len(train_blocks))
print('Test rows :', len(test_df), '| Test blocks :', len(test_blocks))
print('Train share:', round(len(train_df)/len(work_df)*100, 2), '%')
print('Test share :', round(len(test_df)/len(work_df)*100, 2), '%')

In [ ]:
# 12. ສ້າງ Group-aware Sampling Function
# ຊ່ວຍຈຳກັດຈຳນວນ Training Rows ຕໍ່ Fold ໃຫ້ Run ໄດ້ໃນ Google Colab.
# ພະຍາຍາມຮັກສາຕົວຢ່າງຈາກທຸກ Spatial Block.

def sample_by_spatial_group(frame, max_rows, group_col='spatial_block_id', random_state=SEED):
    if max_rows is None or len(frame) <= max_rows:
        return frame.copy()
    sizes = frame[group_col].value_counts().sort_index()
    n_groups = len(sizes)
    if max_rows < n_groups:
        raise ValueError('max_rows ນ້ອຍກວ່າຈຳນວນ Spatial Blocks.')

    raw = sizes / sizes.sum() * max_rows
    quotas = np.floor(raw).astype(int).clip(lower=1)
    quotas = pd.Series(np.minimum(quotas.values, sizes.values), index=sizes.index, dtype=int)

    # ປັບ quota ໃຫ້ຜົນລວມເທົ່າ max_rows.
    while int(quotas.sum()) > max_rows:
        can_reduce = quotas[quotas > 1]
        if can_reduce.empty:
            break
        idx = can_reduce.idxmax()
        quotas.loc[idx] -= 1
    while int(quotas.sum()) < max_rows:
        capacity = sizes - quotas
        capacity = capacity[capacity > 0]
        if capacity.empty:
            break
        idx = capacity.idxmax()
        quotas.loc[idx] += 1

    parts = []
    for group_id, group in frame.groupby(group_col, sort=False):
        n_take = int(quotas.loc[group_id])
        parts.append(group.sample(n=n_take, random_state=random_state))
    sampled = pd.concat(parts, axis=0)
    return sampled.sample(frac=1, random_state=random_state).reset_index(drop=True)


def regression_metrics(y_true, y_pred):
    pred = np.clip(np.asarray(y_pred, dtype=float), 0, 100)
    truth = np.asarray(y_true, dtype=float)
    return {
        'MAE_points': mean_absolute_error(truth, pred),
        'RMSE_points': float(np.sqrt(mean_squared_error(truth, pred))),
        'R2': r2_score(truth, pred),
    }

print('Sampling helper ready.')

## 6. Model Benchmarking

ຈະປຽບທຽບ 5 Models ໃນ Spatial Cross-Validation:

1. **Dummy Regressor:** Baseline ທີ່ທຳນາຍຄ່າສະເລ່ຍ.
2. **Ridge Regression:** Linear baseline ມີ Regularization.
3. **Random Forest Regressor:** ຮອງຮັບ Non-linear relationships.
4. **HistGradientBoosting Regressor:** Gradient boosted trees ທີ່ເໝາະກັບ Tabular Data.
5. **XGBoost Regressor:** Gradient boosting ອີກວິທີໜຶ່ງເພື່ອປຽບທຽບ.

ການເລືອກ Model ໃຊ້ Mean Spatial CV RMSE ທີ່ຕ່ຳສຸດ. ທຸກ Model ຈະໄດ້ຮັບ Train/Validation folds ຊຸດດຽວກັນ.

In [ ]:
# 13. ກຳນົດ Model Factories

MODEL_FACTORIES = {
    'DummyMean': lambda: DummyRegressor(strategy='mean'),
    'Ridge': lambda: Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=10.0))
    ]),
    'RandomForest': lambda: RandomForestRegressor(
        n_estimators=140, max_depth=18, min_samples_leaf=3,
        max_features=0.85, n_jobs=-1, random_state=SEED
    ),
    'HistGradientBoosting': lambda: HistGradientBoostingRegressor(
        max_iter=180, learning_rate=0.07, max_leaf_nodes=31,
        l2_regularization=1.0, early_stopping=True,
        validation_fraction=0.1, random_state=SEED
    ),
    'XGBoost': lambda: XGBRegressor(
        n_estimators=260, max_depth=6, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85,
        reg_alpha=0.0, reg_lambda=2.0,
        objective='reg:squarederror', eval_metric='rmse',
        tree_method='hist', n_jobs=-1, random_state=SEED
    ),
}

print('Models:', list(MODEL_FACTORIES.keys()))

In [ ]:
# 14. ດຳເນີນ 5-Fold Spatial Cross-Validation
# ຖ້າ Runtime ຈຳກັດ ສາມາດປັບ CV_FOLDS ເປັນ 3 ຫຼືຫຼຸດ MAX_CV_TRAIN_ROWS.

CV_FOLDS = 5
MAX_CV_TRAIN_ROWS = 40000

gkf = GroupKFold(n_splits=min(CV_FOLDS, train_df['spatial_block_id'].nunique()))
cv_rows = []

for fold_no, (fit_idx, val_idx) in enumerate(
    gkf.split(train_df, train_df[TARGET_COLUMN], groups=train_df['spatial_block_id']), start=1
):
    fold_fit = train_df.iloc[fit_idx].copy()
    fold_val = train_df.iloc[val_idx].copy()
    fold_fit_sample = sample_by_spatial_group(
        fold_fit, MAX_CV_TRAIN_ROWS, random_state=SEED + fold_no
    )

    X_fit = fold_fit_sample[MODEL_FEATURES]
    y_fit = fold_fit_sample[TARGET_COLUMN]
    X_val = fold_val[MODEL_FEATURES]
    y_val = fold_val[TARGET_COLUMN]

    print(f'Fold {fold_no}: fit={len(fold_fit_sample):,} sampled rows; validation={len(fold_val):,} rows')
    for model_name, factory in MODEL_FACTORIES.items():
        estimator = factory()
        estimator.fit(X_fit, y_fit)
        pred = estimator.predict(X_val)
        metrics = regression_metrics(y_val, pred)
        cv_rows.append({
            'model': model_name, 'fold': fold_no,
            'fit_rows': len(fold_fit_sample), 'validation_rows': len(fold_val),
            **metrics
        })
        print(f"  {model_name:22s} RMSE={metrics['RMSE_points']:.3f} | MAE={metrics['MAE_points']:.3f} | R2={metrics['R2']:.4f}")

cv_results = pd.DataFrame(cv_rows)
cv_results.to_csv(OUTPUT_DIR / 'spatial_cv_metrics_by_fold.csv', index=False)
print('CV complete.')

In [ ]:
# 15. ສະຫຼຸບ Spatial CV ແລະເລືອກ Model ທີ່ດີສຸດ

cv_summary = (
    cv_results.groupby('model')
    .agg(
        Mean_MAE=('MAE_points', 'mean'), SD_MAE=('MAE_points', 'std'),
        Mean_RMSE=('RMSE_points', 'mean'), SD_RMSE=('RMSE_points', 'std'),
        Mean_R2=('R2', 'mean'), SD_R2=('R2', 'std')
    )
    .sort_values('Mean_RMSE')
    .reset_index()
)
display(cv_summary)

best_model_name = cv_summary.iloc[0]['model']
print('Selected model by lowest mean spatial CV RMSE:', best_model_name)

## 7. Final Held-out Spatial Test

ຫຼັງຈາກເລືອກ Model ດ້ວຍ Spatial CV ແລ້ວ, ຈະ Train Model ດ້ວຍ Train Blocks ແລະທົດສອບໃນ Test Blocks ທີ່ Model ບໍ່ເຄີຍເຫັນ.

**ການລາຍງານ:** ຜົນທົດສອບນີ້ຄືຄວາມສາມາດໃນການຈຳລອງ `target_y` ໃນ spatial blocks ທີ່ຖືກກັນໄວ້. ຢ່າລາຍງານເປັນ Real Arabica Suitability Accuracy.

In [ ]:
# 16. Train Best Model ແລະ Evaluate ໃນ Held-out Spatial Test

MAX_FINAL_TEST_TRAIN_ROWS = 120000
train_sample = sample_by_spatial_group(
    train_df, MAX_FINAL_TEST_TRAIN_ROWS, random_state=SEED + 100
)
selected_model = MODEL_FACTORIES[best_model_name]()
selected_model.fit(train_sample[MODEL_FEATURES], train_sample[TARGET_COLUMN])

test_pred = np.clip(selected_model.predict(test_df[MODEL_FEATURES]), 0, 100)
test_metrics = regression_metrics(test_df[TARGET_COLUMN], test_pred)
test_metrics_df = pd.DataFrame([{
    'model': best_model_name,
    'train_rows_used': len(train_sample),
    'test_rows': len(test_df),
    'test_blocks': test_df['spatial_block_id'].nunique(),
    **test_metrics
}])
display(test_metrics_df)
test_metrics_df.to_csv(OUTPUT_DIR / 'heldout_spatial_test_metrics.csv', index=False)

print('Interpretation: metrics measure agreement with the weighted rule-based target, not real-world suitability.')

In [ ]:
# 17. ກວດ Actual vs Predicted ແລະ Residuals

residuals = test_df[TARGET_COLUMN].to_numpy() - test_pred
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(test_df[TARGET_COLUMN], test_pred, s=5, alpha=0.25)
axes[0].plot([0, 100], [0, 100], linestyle='--')
axes[0].set_xlabel('Rule-based target_y')
axes[0].set_ylabel('ML prediction')
axes[0].set_title('Held-out spatial test: actual vs predicted index')
axes[0].set_xlim(0, 100); axes[0].set_ylim(0, 100)

axes[1].hist(residuals, bins=35, edgecolor='white')
axes[1].axvline(0, linestyle='--')
axes[1].set_xlabel('Residual (target_y − prediction)')
axes[1].set_ylabel('Number of cells')
axes[1].set_title('Held-out spatial test residuals')
plt.tight_layout(); plt.show()

In [ ]:
# 18. Permutation Feature Importance ໃນ Held-out Spatial Test
# ຄ່າ Importance ແມ່ນຜົນຕໍ່ການຈຳລອງ Rule Target, ບໍ່ແມ່ນ causal importance.

perm_n = min(6000, len(test_df))
perm_sample = test_df.sample(n=perm_n, random_state=SEED)
perm = permutation_importance(
    selected_model,
    perm_sample[MODEL_FEATURES],
    perm_sample[TARGET_COLUMN],
    scoring='neg_root_mean_squared_error',
    n_repeats=4, random_state=SEED, n_jobs=-1
)
importance_df = pd.DataFrame({
    'feature': MODEL_FEATURES,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)
display(importance_df)
importance_df.to_csv(OUTPUT_DIR / 'permutation_feature_importance.csv', index=False)

plt.figure(figsize=(8, 5))
plt.barh(importance_df['feature'][::-1], importance_df['importance_mean'][::-1],
         xerr=importance_df['importance_std'][::-1])
plt.xlabel('Decrease in score when feature is permuted')
plt.title('Permutation importance — pseudo-target emulation')
plt.tight_layout(); plt.show()

## 8. Train Final Model ສຳລັບສ້າງ National Prediction Layer

ເມື່ອໄດ້ທົດສອບແບບ Spatial Holdout ແລ້ວ, ຈະ Train Final Model ໃໝ່ໂດຍໃຊ້ Dataset ທັງໝົດ (ຫຼື group-aware sample ຖ້າຈຳນວນ Rows ເກີນ Limit). Model ນີ້ໃຊ້ສ້າງ Prediction ສຳລັບທຸກ Grid Cell ໃນ Laos.

ຄ່າ `target_y` ຈາກ Weighted Rule ຈະຖືກຮັກສາໄວ້ຄູ່ກັບ `ml_predicted_suitability` ເພື່ອໃຫ້ປຽບທຽບໄດ້. ຖ້າ ML prediction ແຕກຕ່າງຈາກ rule target ຫຼາຍ ໃຫ້ກວດວ່າ Model ກຳລັງ underfit ຫຼືມີ pattern ທີ່ມັນຈຳລອງບໍ່ໄດ້.

In [ ]:
# 19. Fit Final Model ດ້ວຍ Dataset ທັງໝົດ ແລະສ້າງ Prediction ທຸກ Grid Cell

MAX_FINAL_TRAIN_ROWS = 160000  # ປັບເພີ່ມ/ຫຼຸດຕາມ RAM ຂອງ Colab.
final_train_sample = sample_by_spatial_group(
    work_df, MAX_FINAL_TRAIN_ROWS, random_state=SEED + 500
)
final_model = MODEL_FACTORIES[best_model_name]()
final_model.fit(final_train_sample[MODEL_FEATURES], final_train_sample[TARGET_COLUMN])

work_df['ml_predicted_suitability'] = np.clip(
    final_model.predict(work_df[MODEL_FEATURES]), 0, 100
)
work_df['ml_suitability_band'] = classify_suitability(work_df['ml_predicted_suitability'])
work_df['rule_vs_ml_abs_difference'] = (
    work_df['target_y'] - work_df['ml_predicted_suitability']
).abs()

print('Final model training rows:', len(final_train_sample))
print('All-grid ML prediction summary:')
display(work_df['ml_predicted_suitability'].describe().to_frame())

In [ ]:
# 20. ສ້າງ Map ຄະແນນ Suitability ຈາກ Final ML Model
# ແຜນທີ່ແມ່ນ grid-cell point rendering ຈາກ Dataset V2; ບໍ່ແມ່ນ farm boundary map.

fig, ax = plt.subplots(figsize=(8, 9))
sc = ax.scatter(
    work_df['lon'], work_df['lat'],
    c=work_df['ml_predicted_suitability'],
    cmap='RdYlGn', vmin=0, vmax=100,
    s=1.2, linewidths=0, rasterized=True
)
cb = plt.colorbar(sc, ax=ax, shrink=0.75)
cb.set_label('Predicted suitability index (0–100)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Arabica suitability screening — Lao PDR\nFAO GAUL 2024 AOI · Dataset V2')
ax.set_aspect('equal', adjustable='box')
ax.grid(alpha=0.2)
plt.tight_layout()
map_path = OUTPUT_DIR / 'Arabica_Suitability_Prediction_Map.png'
plt.savefig(map_path, dpi=250, bbox_inches='tight')
plt.show()
print('Map saved to:', map_path)

## 9. Export Model, Score Table ແລະ Documentation

Output ຈະມີ:

- `LaoAgri_Arabica_Suitability_Predictions.csv.gz` — ແຕ່ລະ Grid Cell ພ້ອມ component scores, WLC diagnostic, target ແລະ ML prediction.
- `spatial_cv_metrics_by_fold.csv` ແລະ `spatial_cv_summary.csv` — ຜົນປຽບທຽບ 5-Fold Spatial CV.
- `heldout_spatial_test_metrics.csv` — ຜົນ Final Spatial Holdout.
- `arabica_suitability_weights.csv` — ນ້ຳໜັກ 8 active factors (ລວມ 100%).
- `thermal_altitude_weight_sensitivity.csv` — ປຽບທຽບ group weight 35/40/45%.
- `weight_sensitivity_summary.csv` — one-at-a-time ±20% ສຳລັບ factors ອື່ນ.
- `geometric_score_floor_sensitivity.csv` — ກວດ influence ຂອງ score floor 1/5/10.
- `component_score_saturation_diagnostic.csv` — ກວດ component ທີ່ອາດ score saturation.
- `suitability_class_distribution.csv` — ຈຳນວນ/ສັດສ່ວນ Low/Fair/Good/Very Good.
- `vif_diagnostic.csv` — VIF ຂອງ Model Predictors.
- `permutation_feature_importance.csv`, `final_suitability_model.joblib`, metadata JSON ແລະ map PNG.

**ຂອບເຂດ:** Outputs ນີ້ແມ່ນຜົນຂອງ provisional index ແລະ model ທີ່ຈຳລອງ index; ບໍ່ແມ່ນ field-validated Arabica map.

In [ ]:
# 21. Export Files ແລະ Model Metadata — V3

weights_export = pd.DataFrame([
    {'factor': k, 'weight_percent': v,
     'group': ('Thermal–Altitude' if k in ['elevation','temperature'] else
               'Water' if k in ['rainfall','bio14'] else
               'Soil' if k in ['soil_ph','soc'] else 'Terrain')}
    for k, v in WEIGHTS.items()
])
weights_export.to_csv(OUTPUT_DIR / 'arabica_suitability_weights.csv', index=False)
cv_summary.to_csv(OUTPUT_DIR / 'spatial_cv_summary.csv', index=False)

# Recalculate final class distribution after all model predictions.
class_distribution_export = (
    work_df['suitability_band'].value_counts().reindex(BAND_LABELS, fill_value=0).rename('cells').to_frame()
)
class_distribution_export['percent'] = class_distribution_export['cells'] / len(work_df) * 100
class_distribution_export.to_csv(OUTPUT_DIR / 'suitability_class_distribution.csv')

output_columns = [
    'grid_id', 'lon', 'lat',
    'elevation_m', 'temperature_c', 'rainfall_mm_year', 'soil_ph', 'slope_pct',
    'bio14_driest_month_mm', 'bio15_seasonality_cv', 'soc_0_30cm_g_kg',
    'aspect_deg', 'aspect_sin', 'aspect_cos', 'aspect_is_defined',
    'soil_ph_was_missing', 'soc_was_missing',
    *component_cols, 'target_y_wlc', 'target_y', 'suitability_band', 'limiting_feature',
    'ml_predicted_suitability', 'ml_suitability_band', 'rule_vs_ml_abs_difference'
]

prediction_path = OUTPUT_DIR / 'LaoAgri_Arabica_Suitability_Predictions_V3.csv.gz'
work_df[output_columns].to_csv(prediction_path, index=False, compression='gzip')

model_path = OUTPUT_DIR / 'final_suitability_model_v3.joblib'
joblib.dump(final_model, model_path)

metadata = {
    'project': 'LaoAgri-Suitability',
    'model_name': 'Suitability Model for Predicting Where to Plant Arabica',
    'method_version': 'V3 revised component scores and weighted geometric aggregation',
    'study_area': 'Lao PDR',
    'spatial_reference': 'FAO GAUL 2024 (used to define Dataset V2 AOI)',
    'dataset': str(DATA_PATH),
    'dataset_rows': int(len(work_df)),
    'dataset_columns': list(df_raw.columns),
    'target': 'target_y',
    'target_definition': 'Weighted geometric mean of 8 component scores, 0-100; pseudo-target, not ground truth',
    'aggregation': 'target_y = 100 * exp(sum((weight_i/100) * log(max(component_score_i, 1)/100)))',
    'score_floor_points': SCORE_FLOOR,
    'weights_version': 'Thermal–Altitude group 40%; Temperature 28%, Elevation 12%',
    'weights_percent': {k: float(v) for k, v in WEIGHTS.items()},
    'weight_groups': {
        'thermal_altitude': 40.0,
        'water': 25.0,
        'soil': 20.0,
        'terrain': 15.0,
        'bio15': 'diagnostic_only'
    },
    'score_bands': {'Low': '[0,50)', 'Fair': '[50,70)', 'Good': '[70,85)', 'Very Good': '[85,100]'},
    'soc_score_method': 'Percentile rank within Lao PDR Dataset V2; missing SOC receives neutral score 50; relative indicator only',
    'bio15_role': 'Diagnostic and ML predictor only; not included in target because no justified monotonic/optimum direction was specified',
    'model_features': MODEL_FEATURES,
    'selected_model': best_model_name,
    'spatial_block_size_degrees': BLOCK_SIZE_DEG,
    'random_seed': SEED,
    'missing_value_medians': {'soil_ph': PH_MEDIAN, 'soc_0_30cm_g_kg': SOC_MEDIAN},
    'spatial_cv_summary': cv_summary.to_dict(orient='records'),
    'heldout_spatial_test_metrics': test_metrics,
    'limitations': [
        'Metrics measure emulation of the rule-based pseudo-target, not real-world Arabica suitability or yield.',
        'FAO GAUL 2024 is an administrative boundary source, not a crop-requirement or weighting standard.',
        'FAO Arabica Coffee Manual for Lao PDR is the main local agronomic reference for site conditions.',
        'Weights and several score curve interpolation points are researcher assumptions and require independent validation.',
        'Elevation and temperature are strongly correlated; grouping them into 40% reduces double-counting but does not remove statistical redundancy in ML predictors.',
        'VIF is a diagnostic of predictor redundancy and is not a method for calculating agronomic weights.',
        'SOC score is a within-Laos percentile rank and is not an absolute agronomic suitability threshold; it is not directly portable to another AOI without a reference distribution.',
        'BIO15 is excluded from target scoring because no defensible direction of suitability was specified; it remains a diagnostic and model input.',
        'BIO14 is a single driest-month proxy and does not measure dry-season duration or drought severity.',
        'The weighted geometric aggregation is less compensatory than WLC; score floor=1 is a modelling assumption with sensitivity analysis.',
        'Project bands Low/Fair/Good/Very Good are descriptive index ranges, not FAO S1/S2/S3/N classes.',
        'WorldClim 2.1 climate normals use 1970-2000 baseline.',
        'Spatial grid is approximately 928 m and is not suitable for farm-plot decisions.',
        'Slope is a grid-cell mean derived from a DSM; it is not a plot-level slope measurement.',
        'SoilGrids pH and SOC are spatial predictions; median-imputed cells are flagged.'
    ],
    'references': {
        'FAO_GAUL_2024': 'https://www.fao.org/agroinformatics/training-and-resources/data-sets/data-set-detail/global-gaul-new-2024-release/en',
        'FAO_Arabica_Coffee_Manual_Lao_PDR_Chapter_1': 'https://www.fao.org/4/ae939e/ae939e03.htm',
        'FAO_Arabica_Coffee_Manual_Lao_PDR_Chapter_6': 'https://www.fao.org/4/ae939e/ae939e06.htm',
        'FAO_EcoCrop_Coffea_arabica': 'https://ecocrop.apps.fao.org/ecocrop/srv/en/dataSheet?id=749',
        'Gruter_2022': 'https://doi.org/10.1371/journal.pone.0261976',
        'Ovalle_Rivera_2015': 'https://doi.org/10.1371/journal.pone.0124155'
    }
}
with open(OUTPUT_DIR / 'suitability_model_metadata_v3.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print('Saved:')
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(f'{p.name:55s} {p.stat().st_size/1024/1024:8.2f} MB')

In [ ]:
# 22. ລວມ Outputs ເປັນ ZIP ແລະດາວໂຫຼດ

zip_path = OUTPUT_DIR / 'Suitability_Model_for_Predicting_Where_to_Plant_Arabica_Outputs.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.iterdir():
        if p.is_file() and p.name != zip_path.name:
            zf.write(p, arcname=p.name)

print('ZIP created:', zip_path)
files.download(str(zip_path))

## 10. ການອ່ານຜົນ ແລະຂໍ້ຈຳກັດທາງວິຊາການ

### ຄວນຕີຄວາມແນວໃດ?

- `target_y`: Weighted Geometric Arabica Suitability Screening Index (0–100) ຈາກ 8 active component scores.
- `target_y_wlc`: Weighted Linear Combination ທີ່ເກັບໄວ້ເພື່ອປຽບທຽບ aggregation method.
- `ml_predicted_suitability`: ຄະແນນທີ່ Regression Model ຮຽນຮູ້ເພື່ອຈຳລອງ `target_y`.
- `suitability_band`: Low 0–49, Fair 50–69, Good 70–84, Very Good 85–100. ເປັນ project-defined bands ເທົ່ານັ້ນ.
- `limiting_feature`: component ທີ່ໄດ້ຄະແນນຕ່ຳສຸດຂອງ Grid Cell; ເປັນ diagnostic.

### ຂໍ້ຈຳກັດສຳຄັນ

1. `target_y` ແມ່ນ pseudo-target ທີ່ສ້າງຈາກ X ບໍ່ແມ່ນ field observation. Model metrics ຈຶ່ງບໍ່ແມ່ນ real-world accuracy.
2. VIF ສູງຂອງ Elevation/Temperature ເປັນບັນຫາ predictor redundancy. ການຈັດທັງສອງເປັນ group 40% ແມ່ນວິທີຄວບຄຸມ Weight ໃນ scoring index; ບໍ່ໄດ້ລຶບ multicollinearity ອອກຈາກ ML inputs.
3. SOC percentile rank ອະທິບາຍພຽງລຳດັບ relative ໃນ Laos Dataset V2; ບໍ່ໃຊ້ເທົ່າກັບ threshold ຂອງ soil organic matter ຈາກຄູ່ມື.
4. pH ແລະ annual rainfall ອາດຍັງມີຫຼາຍພື້ນທີ່ໄດ້ component scores ສູງ. ຖ້າຄ່າດິນ/ຝົນທົ່ວປະເທດຢູ່ໃນຊ່ວງທີ່ຄູ່ມືລະບຸວ່າເໝາະ, ການສະແດງຄະແນນສູງນັ້ນບໍ່ຄວນຖືກລຶບທິ້ງໂດຍບໍ່ມີຫຼັກຖານ.
5. BIO15 ເປັນ diagnostic-only ໃນ Target; ມັນຍັງຢູ່ໃນ ML predictors ເພື່ອທົດສອບການຈຳລອງ Target.
6. BIO14 ບອກພຽງ precipitation ຂອງເດືອນແຫ້ງທີ່ສຸດ; ບໍ່ໄດ້ວັດ dry-season duration ຫຼື PET.
7. Weighted geometric mean ຫຼຸດການຊົດເຊີຍລະຫວ່າງ factors ແຕ່ກໍສ້າງ score ທີ່ເຄັ່ງກວ່າ WLC. ຕ້ອງອ່ານ Floor Sensitivity ປະກອບ.
8. FAO Lao manual ແລະ FAO EcoCrop ບໍ່ໄດ້ກຳນົດຕົວເລກ Weights ຂອງ V3; weights ແມ່ນ project assumptions.
9. WorldClim 2.1 ໃຊ້ climate baseline 1970–2000; ບໍ່ແມ່ນ 2026 climate observation.
10. ຄວາມລະອຽດປະມານ 928 m ແມ່ນເໝາະສຳລັບ national/regional screening, ບໍ່ແມ່ນການເລືອກຈຸດປູກລະດັບແປງ.

### References

1. **FAO.** *Arabica Coffee Manual for Lao PDR* (2005), Chapter 1: Coffee Plant & Site Selection. https://www.fao.org/4/ae939e/ae939e03.htm
2. **FAO.** *Arabica Coffee Manual for Lao PDR* (2005), Chapter 6: Optimum Nutrient Levels. https://www.fao.org/4/ae939e/ae939e06.htm
3. **FAO.** EcoCrop Data Sheet — *Coffea arabica*. https://ecocrop.apps.fao.org/ecocrop/srv/en/dataSheet?id=749
4. **FAO.** Global Administrative Unit Layers (GAUL) 2024 release. https://www.fao.org/agroinformatics/training-and-resources/data-sets/data-set-detail/global-gaul-new-2024-release/en
5. **Grüter, R. et al. (2022).** Coffee suitability criteria / land suitability assessment reference. PLOS ONE 17(1), e0261976. https://doi.org/10.1371/journal.pone.0261976
6. **Ovalle-Rivera, O. et al. (2015).** Projected shifts in Coffea arabica suitability among major global producing regions due to climate change. PLOS ONE 10(4), e0124155. https://doi.org/10.1371/journal.pone.0124155

**ຫຼັກການ:** FAO GAUL 2024 ກຳນົດຂອບເຂດພູມສາດ; FAO Lao Arabica Manual ແລະ FAO EcoCrop ຊ່ວຍອ້າງອີງ crop requirements; weights ແລະ curve interpolation ຂອງ V3 ແມ່ນ assumptions ທີ່ຕ້ອງການ sensitivity ແລະ independent validation.